<a href="https://colab.research.google.com/github/atreatul1-sketch/Python-Project/blob/main/UnifiedMentorProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# ==============================
# CONFIGURATION
# ==============================
INPUT_FILE = "Nassau Candy Distributor.csv"

OUTPUT_CLEAN_FILE = "cleaned_shipments.csv"
OUTPUT_ROUTE_STATE_FILE = "route_state_summary.csv"
OUTPUT_ROUTE_REGION_FILE = "route_region_summary.csv"
OUTPUT_STATE_FILE = "state_summary.csv"
OUTPUT_REGION_FILE = "region_summary.csv"
OUTPUT_SHIPMODE_FILE = "shipmode_summary.csv"
OUTPUT_KPI_FILE = "kpi_summary.csv"
OUTPUT_TOP10_FILE = "top_10_routes.csv"
OUTPUT_BOTTOM10_FILE = "bottom_10_routes.csv"
OUTPUT_BOTTLENECK_FILE = "bottleneck_routes.csv"

DELAY_THRESHOLD_DAYS = 5

# Dataset columns
COL_ORDER_DATE = "Order Date"
COL_SHIP_DATE = "Ship Date"
COL_SHIP_MODE = "Ship Mode"
COL_COUNTRY = "Country/Region"
COL_CITY = "City"
COL_STATE = "State/Province"
COL_REGION = "Region"
COL_DIVISION = "Division"
COL_CUSTOMER_ID = "Customer ID"

# ==============================
# LOAD DATA
# ==============================
df = pd.read_csv(INPUT_FILE)
df.columns = [c.strip() for c in df.columns]

# ==============================
# VALIDATE REQUIRED COLUMNS
# ==============================
required_cols = [
    COL_ORDER_DATE, COL_SHIP_DATE, COL_SHIP_MODE,
    COL_COUNTRY, COL_CITY, COL_STATE, COL_REGION,
    COL_DIVISION, COL_CUSTOMER_ID
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# ==============================
# DATA CLEANING
# ==============================
df[COL_ORDER_DATE] = pd.to_datetime(df[COL_ORDER_DATE], errors="coerce")
df[COL_SHIP_DATE] = pd.to_datetime(df[COL_SHIP_DATE], errors="coerce")

df = df.dropna(subset=required_cols).copy()
df = df.dropna(subset=[COL_ORDER_DATE, COL_SHIP_DATE]).copy()

for col in [COL_SHIP_MODE, COL_COUNTRY, COL_CITY, COL_STATE, COL_REGION, COL_DIVISION]:
    df[col] = df[col].astype(str).str.strip().str.title()

df["Shipping Lead Time"] = (df[COL_SHIP_DATE] - df[COL_ORDER_DATE]).dt.days
df = df[df["Shipping Lead Time"].notna()].copy()
df = df[df["Shipping Lead Time"] >= 0].copy()

df["Delayed"] = (df["Shipping Lead Time"] > DELAY_THRESHOLD_DAYS).astype(int)

# Save cleaned data
df.to_csv(OUTPUT_CLEAN_FILE, index=False)

# ==============================
# FEATURE ENGINEERING
# ==============================
# Using Division as a proxy for factory/business unit
df["Route_State"] = df[COL_DIVISION].astype(str) + " -> " + df[COL_STATE].astype(str)
df["Route_Region"] = df[COL_DIVISION].astype(str) + " -> " + df[COL_REGION].astype(str)

# ==============================
# ROUTE AGGREGATION
# ==============================
def summarize_routes(group_col):
    summary = (
        df.groupby(group_col)
          .agg(
              Total_Shipments=("Shipping Lead Time", "size"),
              Avg_Lead_Time=("Shipping Lead Time", "mean"),
              Lead_Time_Variability=("Shipping Lead Time", "std"),
              Delay_Frequency=("Delayed", "mean")
          )
          .reset_index()
    )
    summary["Lead_Time_Variability"] = summary["Lead_Time_Variability"].fillna(0)

    # Normalized efficiency score: higher is better
    lt_norm = 1 - (summary["Avg_Lead_Time"] - summary["Avg_Lead_Time"].min()) / (
        summary["Avg_Lead_Time"].max() - summary["Avg_Lead_Time"].min() + 1e-9
    )
    var_norm = 1 - (summary["Lead_Time_Variability"] - summary["Lead_Time_Variability"].min()) / (
        summary["Lead_Time_Variability"].max() - summary["Lead_Time_Variability"].min() + 1e-9
    )
    vol_norm = (summary["Total_Shipments"] - summary["Total_Shipments"].min()) / (
        summary["Total_Shipments"].max() - summary["Total_Shipments"].min() + 1e-9
    )

    summary["Route_Efficiency_Score"] = 0.5 * lt_norm + 0.3 * var_norm + 0.2 * vol_norm
    return summary.sort_values(
        by=["Avg_Lead_Time", "Lead_Time_Variability", "Delay_Frequency"],
        ascending=[True, True, True]
    )

route_state_summary = summarize_routes("Route_State")
route_region_summary = summarize_routes("Route_Region")

top_10_routes = route_state_summary.head(10).copy()
bottom_10_routes = route_state_summary.tail(10).copy()

# ==============================
# GEOGRAPHIC BOTTLENECK ANALYSIS
# ==============================
def summarize_geo(group_col):
    summary = (
        df.groupby(group_col)
          .agg(
              Total_Shipments=("Shipping Lead Time", "size"),
              Avg_Lead_Time=("Shipping Lead Time", "mean"),
              Lead_Time_Variability=("Shipping Lead Time", "std"),
              Delay_Frequency=("Delayed", "mean")
          )
          .reset_index()
          .sort_values(by=["Avg_Lead_Time", "Total_Shipments"], ascending=[False, False])
    )
    summary["Lead_Time_Variability"] = summary["Lead_Time_Variability"].fillna(0)
    return summary

state_summary = summarize_geo(COL_STATE)
region_summary = summarize_geo(COL_REGION)

# High volume + poor performance bottlenecks
volume_threshold = route_state_summary["Total_Shipments"].quantile(0.75)
leadtime_threshold = route_state_summary["Avg_Lead_Time"].quantile(0.75)

bottleneck_routes = route_state_summary[
    (route_state_summary["Total_Shipments"] >= volume_threshold) &
    (route_state_summary["Avg_Lead_Time"] >= leadtime_threshold)
].copy()

# ==============================
# SHIP MODE PERFORMANCE
# ==============================
shipmode_summary = (
    df.groupby(COL_SHIP_MODE)
      .agg(
          Total_Shipments=("Shipping Lead Time", "size"),
          Avg_Lead_Time=("Shipping Lead Time", "mean"),
          Lead_Time_Variability=("Shipping Lead Time", "std"),
          Delay_Frequency=("Delayed", "mean")
      )
      .reset_index()
      .sort_values(by="Avg_Lead_Time", ascending=True)
)
shipmode_summary["Lead_Time_Variability"] = shipmode_summary["Lead_Time_Variability"].fillna(0)

# Optional comparison of standard vs expedited
if set(["Standard Class", "Second Class", "First Class", "Same Day"]).intersection(set(df[COL_SHIP_MODE].unique())):
    print("\nShip mode comparison available.")
else:
    print("\nShip mode values may differ from common labels (Standard Class, Second Class, First Class, Same Day).")

# ==============================
# KPI SUMMARY
# ==============================
kpi_summary = pd.DataFrame({
    "KPI": [
        "Shipping Lead Time",
        "Average Lead Time",
        "Route Volume",
        "Delay Frequency",
        "Route Efficiency Score"
    ],
    "Description": [
        "Ship Date - Order Date",
        "Mean shipping duration per route",
        "Number of orders per route",
        "% of shipments exceeding threshold",
        "Normalized lead-time performance"
    ],
    "Value": [
        df["Shipping Lead Time"].mean(),
        route_state_summary["Avg_Lead_Time"].mean(),
        route_state_summary["Total_Shipments"].sum(),
        df["Delayed"].mean(),
        route_state_summary["Route_Efficiency_Score"].mean()
    ]
})

# ==============================
# EXPORT RESULTS
# ==============================
route_state_summary.to_csv(OUTPUT_ROUTE_STATE_FILE, index=False)
route_region_summary.to_csv(OUTPUT_ROUTE_REGION_FILE, index=False)
state_summary.to_csv(OUTPUT_STATE_FILE, index=False)
region_summary.to_csv(OUTPUT_REGION_FILE, index=False)
shipmode_summary.to_csv(OUTPUT_SHIPMODE_FILE, index=False)
kpi_summary.to_csv(OUTPUT_KPI_FILE, index=False)
top_10_routes.to_csv(OUTPUT_TOP10_FILE, index=False)
bottom_10_routes.to_csv(OUTPUT_BOTTOM10_FILE, index=False)
bottleneck_routes.to_csv(OUTPUT_BOTTLENECK_FILE, index=False)

# ==============================
# PRINT RESULTS
# ==============================
print("\nTop 10 Most Efficient Routes:")
print(top_10_routes[["Route_State", "Total_Shipments", "Avg_Lead_Time", "Lead_Time_Variability", "Delay_Frequency", "Route_Efficiency_Score"]])

print("\nBottom 10 Least Efficient Routes:")
print(bottom_10_routes[["Route_State", "Total_Shipments", "Avg_Lead_Time", "Lead_Time_Variability", "Delay_Frequency", "Route_Efficiency_Score"]])

print("\nState Summary:")
print(state_summary.head(10))

print("\nRegion Summary:")
print(region_summary.head(10))

print("\nShip Mode Summary:")
print(shipmode_summary)

print("\nKPI Summary:")
print(kpi_summary)

print("\nBottleneck Routes:")
print(bottleneck_routes)

/tmp/ipykernel_812/1867947684.py:55: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[COL_SHIP_DATE] = pd.to_datetime(df[COL_SHIP_DATE], errors="coerce")



Ship mode comparison available.

Top 10 Most Efficient Routes:
                Route_State  Total_Shipments  Avg_Lead_Time  \
27     Chocolate -> Montana                3          704.0   
75          Other -> Nevada                1          730.0   
85  Other -> South Carolina                1          821.0   
78      Other -> New Mexico                1          904.0   
69       Other -> Louisiana                1          908.0   
88        Other -> Virginia                2          973.0   
62     Other -> Connecticut                1          995.0   
74        Other -> Nebraska                1         1024.0   
19       Chocolate -> Maine                3         1026.0   
73       Other -> Minnesota                1         1056.0   

    Lead_Time_Variability  Delay_Frequency  Route_Efficiency_Score  
27               0.000000              1.0                0.800523  
75               0.000000              1.0                0.788029  
85               0.000000          